In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

In [2]:
def detect_hand_contacts_from_distance(
    hand_distance,
    frame_rate,
    min_gap_sec=0.3,
    min_distance_thresh=0.19,
    prominence=0.01,
    slope_min=0.3
):
    """
    Detect hand-contact events using only inter-hand distance.

    Parameters
    ----------
    hand_distance : np.ndarray
        1D array of inter-hand distance over time
    frame_rate : float
        Sampling rate (frames per second)
    min_gap_sec : float, optional
        Minimum temporal gap between extrema (seconds)
    min_distance_thresh : float, optional
        Keep only minima where hand_distance <= this value
    prominence : float, optional
        Peak prominence for maxima/minima detection
    slope_min : float, optional
        Minimum descent slope (distance units per second)

    Returns
    -------
    contacts : list of int
        Frame indices of detected hand contacts
    """

    min_gap_samples = int(min_gap_sec * frame_rate)

    # --- find maxima ---
    max_idx, _ = find_peaks(
        hand_distance,
        distance=min_gap_samples,
        prominence=prominence
    )

    # --- find minima (invert signal) ---
    min_idx, _ = find_peaks(
        -hand_distance,
        distance=min_gap_samples,
        prominence=prominence,
        height=-min_distance_thresh
    )

    contacts = []

    for t_min in min_idx:
        prev_max = max_idx[max_idx < t_min]
        if len(prev_max) == 0:
            continue

        t_max = prev_max[-1]

        delta_d = hand_distance[t_max] - hand_distance[t_min]
        delta_t = (t_min - t_max) / frame_rate

        if delta_t <= 0:
            continue

        slope = delta_d / delta_t

        if slope >= slope_min:
            contacts.append(t_min)

    return contacts

In [3]:
DIR_CSV = "bvh_to_csv"
frame_rate = 240
refined_window = 0.1

with open('data/selected_piece_list.pkl', 'rb') as f:
        piece_list = pickle.load(f)

In [5]:
save_dir = os.path.join("data", "hand_clap_onsets_05jun2026")
os.makedirs(save_dir, exist_ok=True)

for file_name in piece_list:
    #         pass

    # file_name = piece_list[0]
    # base_name = piece_list[0]

    # Load joint position data for both wrists
    base_name = os.path.splitext(os.path.basename(file_name))[0]
    worldpos_file = os.path.join(DIR_CSV, f"{base_name}_T_worldpos.csv")

    try:
        world_positions = pd.read_csv(worldpos_file)
        print(f"Successfully loaded CSV with {len(world_positions)} rows")
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise

    # Get time column and position data for both wrists
    time_column = world_positions.columns[0]  # First column is time
    times = world_positions[time_column].values

    # Get positions for both wrists along the chosen axis
    L_pos_x = world_positions[f"LeftWrist.X"].values
    R_pos_x = world_positions[f"RightWrist.X"].values

    L_pos_y = world_positions[f"LeftWrist.Y"].values
    R_pos_y = world_positions[f"RightWrist.Y"].values

    L_pos_z = world_positions[f"LeftWrist.Z"].values
    R_pos_z = world_positions[f"RightWrist.Z"].values


    # Calculate velocities
    dt = 1.0 / frame_rate
    L_vel_x = np.gradient(L_pos_x, dt)
    R_vel_x = np.gradient(R_pos_x, dt)

    L_vel_y = np.gradient(L_pos_y, dt)
    R_vel_y = np.gradient(R_pos_y, dt)

    L_vel_z = np.gradient(L_pos_z, dt)
    R_vel_z = np.gradient(R_pos_z, dt)

    rel_vel = np.sqrt(
        (L_vel_x - R_vel_x)**2 +
        (L_vel_y - R_vel_y)**2 +
        (L_vel_z - R_vel_z)**2
    )

    # Calculate accelerations
    L_acc_x = np.gradient(L_vel_x, dt)
    R_acc_x = np.gradient(R_vel_x, dt)

    L_acc_y = np.gradient(L_vel_y, dt)
    R_acc_y = np.gradient(R_vel_y, dt)

    L_acc_z = np.gradient(L_vel_z, dt)
    R_acc_z = np.gradient(R_vel_z, dt)

    rel_acc = np.sqrt(
        (L_acc_x - R_acc_x)**2 +
        (L_acc_y - R_acc_y)**2 +
        (L_acc_z - R_acc_z)**2
    )


    hand_distance = np.sqrt(
        (L_pos_x - R_pos_x)**2 +
        (L_pos_y - R_pos_y)**2 +
        (L_pos_z - R_pos_z)**2
    )


    hand_contacts = detect_hand_contacts_from_distance(
                    hand_distance,
                    frame_rate=frame_rate,
                    min_gap_sec=0.3,
                    min_distance_thresh=0.19*100,
                    slope_min=0.3
                )

    # Choose  minima around the contacts using window from rel_speed or rel_acc
    refined_contacts = []
    win = int(refined_window * frame_rate)   # ± ms window

    file_parts = file_name.split("_")
    ensmble = file_parts[1]

    if ensmble == "E1" or ensmble == "E2":
        acc_threshold = 3000

    if ensmble == "E3":
        acc_threshold = 1500   


    for c in hand_contacts:
        # for rel_acc
        start = max(0, c - win)
        end = min(len(rel_acc), c + int(0.03* frame_rate))
        seg = rel_acc[start:end]
        
        # find local maxima in the window
        peaks, _ = find_peaks(seg, prominence=0.1* np.max(seg))
        if len(peaks) == 0:
            continue

        # convert to absolute indices
        abs_peaks = start + peaks
        peak_vals = seg[peaks]
        
        # keep only peaks above threshold
        mask = peak_vals > acc_threshold
        peak_vals = peak_vals[mask]
        abs_peaks = abs_peaks[mask]
        
        # sort peaks by magnitude (descending)
        order = np.argsort(peak_vals)[::-1]     # order = np.argsort(peak_vals)[::-1]
        abs_peaks = abs_peaks[order]
        peak_vals = peak_vals[order]
        
        # ---- NEW LOGIC ----
        if len(peak_vals) == 0:
            continue  # or skip this window safely

        elif len(peak_vals) == 1:
            local_max = abs_peaks[0]

        else:
            # identify earlier and later in time
            if abs_peaks[0] <= abs_peaks[1]:
                earlier_idx, later_idx = 0, 1
            else:
                earlier_idx, later_idx = 1, 0

            p_earlier = peak_vals[earlier_idx]
            p_later = peak_vals[later_idx]

            # symmetric relative difference
            rel_diff = abs(p_earlier - p_later) / max(p_earlier, p_later)

            # Rule 1: earlier peak stronger → choose earlier
            if p_earlier >= p_later:
                local_max = abs_peaks[earlier_idx]

            # Rule 2: later stronger, but earlier within 30% → choose earlier
            elif rel_diff < 0.30:
                local_max = abs_peaks[earlier_idx]

            # Rule 3: otherwise choose strongest
            else:
                # local_max = abs_peaks[later_idx]
                if p_earlier > p_later:
                    local_max = abs_peaks[earlier_idx]
                else:
                    local_max = abs_peaks[later_idx]
    # 
        refined_contacts.append(local_max)

        # local_max = start + np.argmax(rel_acc[start:end])
        # refined_contacts.append(local_max)
            
    contacts = refined_contacts
    print(f"Detected {len(contacts)} hand contacts")
    if len(contacts) > 0:
        contact_times_all = times[contacts]
        print(f"Contact times range: {contact_times_all.min():.3f}s to {contact_times_all.max():.3f}s")

    #save the contact times as csv
    
    save_path = os.path.join(save_dir, f"{base_name}_hand_clap_onsets.csv")
    pd.DataFrame({"contact_times": contact_times_all}).to_csv(save_path, index=False)
    print(f"Saved hand clap onsets to {save_path}")




Successfully loaded CSV with 107339 rows
Detected 146 hand contacts
Contact times range: 11.809s to 426.047s
Saved hand clap onsets to data/hand_clap_onsets_05jun2026/BKO_E1_D1_01_Suku_hand_clap_onsets.csv
Successfully loaded CSV with 81793 rows
Detected 96 hand contacts
Contact times range: 65.614s to 332.297s
Saved hand clap onsets to data/hand_clap_onsets_05jun2026/BKO_E1_D1_02_Maraka_hand_clap_onsets.csv
Successfully loaded CSV with 80027 rows
Detected 36 hand contacts
Contact times range: 134.965s to 299.678s
Saved hand clap onsets to data/hand_clap_onsets_05jun2026/BKO_E1_D1_03_Wasulunka_hand_clap_onsets.csv
Successfully loaded CSV with 117813 rows
Detected 119 hand contacts
Contact times range: 62.551s to 484.205s
Saved hand clap onsets to data/hand_clap_onsets_05jun2026/BKO_E1_D1_06_Manjanin_hand_clap_onsets.csv
Successfully loaded CSV with 64869 rows
Detected 70 hand contacts
Contact times range: 81.040s to 258.246s
Saved hand clap onsets to data/hand_clap_onsets_05jun2026/BKO